In [3]:
RUN_NAME = "redolent-koi-449"

In [4]:
import os

# os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:9091"
# from datahub_integrations.experimentation.ai_init import AI_EXPERIMENTATION_INITIALIZED

import dotenv

dotenv.load_dotenv("../../src/datahub_integrations/experimentation/.env")
import mlflow

mlflow.set_experiment("docs_generation")

runs = mlflow.search_runs(
    filter_string=f"attributes.run_name  = '{RUN_NAME}'",
    output_format="list",
    order_by=["start_time DESC"],
)
assert len(runs) == 1

## Run details

In [ ]:
print(f"Analysis for Run: {RUN_NAME}")
print(f"Run ID: {runs[0].info.run_id}")
print(f"Run params: {runs[0].data.params}")
print(f"Run metrics: {runs[0].data.metrics}")
print(f"Run tags: {runs[0].data.tags}")

In [ ]:
eval_results_table = mlflow.load_table(
    "eval_results_table.json", run_ids=[runs[0].info.run_id]
)

# print(eval_results_table.columns)

## Total Description Generation Time Distribution

In [ ]:
import matplotlib.pyplot as plt

col_name = "generation_time"


def plot_hist(df, col_name, breakdown_col=None):
    if df is not None and col_name in df.columns:
        # Plot histogram of generation time
        plt.figure(figsize=(10, 6))
        bin_edges = list(range(0, 301, 15))

        if breakdown_col is not None and breakdown_col in df.columns:
            # Create stacked histogram colored by breakdown_col
            categories = df[breakdown_col].unique()
            colors = plt.cm.Set3(
                range(len(categories))
            )  # Use a colormap for distinct colors

            # Create data for each category
            data_by_category = [
                df[df[breakdown_col] == cat][col_name] for cat in categories
            ]

            plt.hist(
                data_by_category,
                bins=bin_edges,
                alpha=0.7,
                color=colors,
                edgecolor="black",
                label=categories,
                stacked=True,
            )
            plt.legend(title=breakdown_col)
        else:
            # Original single histogram
            plt.hist(
                df[col_name],
                bins=bin_edges,
                alpha=0.7,
                color="skyblue",
                edgecolor="black",
            )

        plt.title(f"Distribution of {col_name} (Run ID: {runs[0].info.run_id})")
        plt.xlabel(f"{col_name}")
        plt.ylabel("Frequency")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Print some statistics
        print(f"{col_name} Statistics:")
        print(f"Total rows: {df.shape[0]}")
        print(f"Mean: {df[col_name].mean():.2f}")
        print(f"Median: {df[col_name].median():.2f}")
        print(f"Min: {df[col_name].min():.2f}")
        print(f"Max: {df[col_name].max():.2f}")

        # Print breakdown statistics if breakdown_col is provided
        if breakdown_col is not None and breakdown_col in df.columns:
            print(f"\nBreakdown by {breakdown_col}:")
            for cat in df[breakdown_col].unique():
                cat_data = df[df[breakdown_col] == cat][col_name]
                print(
                    f"  {cat}: {len(cat_data)} rows, Mean: {cat_data.mean():.2f}s, Median: {cat_data.median():.2f}s"
                )
    else:
        print(f"Could not find {col_name} column in the evaluation results.")


plot_hist(eval_results_table, col_name, breakdown_col="has_table_description/score")

## Datasets having table description with less than 100% columns described

In [ ]:
import pandas as pd


tables_lt_100_percent_columns_described = eval_results_table[
    (eval_results_table["has_table_description/score"] == True)
    & (eval_results_table["percent_columns_described"] < 100)
]
print(
    "Total tables with less than 100% columns described: ",
    len(tables_lt_100_percent_columns_described),
)
if len(tables_lt_100_percent_columns_described) > 0:
    pd.set_option("display.max_colwidth", None)
    display(
        tables_lt_100_percent_columns_described[
            ["urn", "total_columns", "percent_columns_described"]
        ]
    )

## Datasets having high overall total generation time

In [ ]:
tables_gt_60s_generation_time = eval_results_table[
    (eval_results_table["has_table_description/score"] == True)
    & (eval_results_table["generation_time"] > 60)
]
if len(tables_gt_60s_generation_time) > 0:
    print(
        "Total tables with generation time greater than 60 seconds: ",
        len(tables_gt_60s_generation_time),
    )
    pd.set_option("display.max_colwidth", None)
    display(
        tables_gt_60s_generation_time[
            ["urn", "generation_time", "total_columns", "percent_columns_described"]
        ]
    )

## Get all traces for mlflow run and flatten as one per span

In [ ]:
import mlflow

# Get all traces for the specific run
traces = mlflow.search_traces(
    experiment_ids=[runs[0].info.experiment_id],
    filter_string=f"attributes.run_id = '{runs[0].info.run_id}'",
    return_type="pandas",
)

print(f"Found {len(traces)} traces for run {RUN_NAME}")

In [ ]:
# Explode traces to have one row per span using pandas native method
import pandas as pd

# First, explode the spans column to create one row per span
traces_exploded = traces.explode("spans")
traces_exploded["urn"] = traces_exploded["tags"].apply(lambda x: x["urn"])
# print(traces_exploded.columns)
# Extract span details into separate columns using pd.json_normalize
if not traces_exploded["spans"].isna().all():
    span_details = pd.json_normalize(traces_exploded["spans"]).add_prefix("span_")

    # Combine the original trace data with span details
    traces_exploded = pd.concat(
        [
            traces_exploded.drop("spans", axis=1).reset_index(drop=True),
            span_details.reset_index(drop=True),
        ],
        axis=1,
    )

else:
    # If no spans data, just add empty span columns
    span_columns = [
        "span_name",
        "span_parent_id",
        "span_start_time",
        "span_end_time",
        "span_status_code",
        "span_status_message",
        "span_events",
        "span_context.span_id",
        "span_context.trace_id",
        "span_attributes.mlflow.traceRequestId",
        "span_attributes.mlflow.spanType",
        "span_attributes.mlflow.spanFunctionName",
        "span_attributes.mlflow.spanInputs",
        "span_attributes.mlflow.spanOutputs",
    ]
    for col in span_columns:
        traces_exploded[col] = None
    traces_exploded = traces_exploded.drop("spans", axis=1)

traces_exploded["span_execution_time_ms"] = (
    traces_exploded["span_end_time"] - traces_exploded["span_start_time"]
) / 1e6
print(f"Original traces: {len(traces)} rows")
print(f"Exploded spans: {len(traces_exploded)} rows")
# print(traces_exploded.columns)

## Trace analysis to find table description generation time

In [ ]:
all_top_spans = traces_exploded[
    traces_exploded["span_name"].str.contains("generate_entity_descriptions_for_urn")
]
top_span_ids = all_top_spans["span_context.span_id"].drop_duplicates().to_list()
top_span_ids
# Filter to only include spans with name starting with 'BedrockRuntime.invoke_model'
bedrock_spans = traces_exploded[
    traces_exploded["span_name"].str.startswith("BedrockRuntime.invoke_model", na=False)
]

# Include only spans with parent span id in top_span_ids
bedrock_spans_table_desc = bedrock_spans[
    bedrock_spans["span_parent_id"].isin(top_span_ids)
]

plot_hist(
    bedrock_spans_table_desc, "span_execution_time_ms", breakdown_col="span_status_code"
)

## Trace analysis of column batch description generation time

In [ ]:
column_batch_top_span_ids = (
    traces_exploded[
        traces_exploded["span_name"].str.contains("generate_column_batch_descriptions")
    ]["span_context.span_id"]
    .drop_duplicates()
    .to_list()
)
failed_bedrock_spans_column_batch = bedrock_spans[
    bedrock_spans["span_parent_id"].isin(column_batch_top_span_ids)
]
plot_hist(
    failed_bedrock_spans_column_batch,
    "span_execution_time_ms",
    breakdown_col="span_status_code",
)

## Trace analysis to find failed bedrock invoke calls

In [ ]:
# Filter to only include spans with name starting with 'BedrockRuntime.invoke_model'
bedrock_spans = traces_exploded[
    traces_exploded["span_name"].str.startswith("BedrockRuntime.invoke_model", na=False)
]
print(f"Total Bedrock invoke spans: {len(bedrock_spans)}")

# Filter failed calls - adjust the condition based on actual status column structure
failed_bedrock_spans = bedrock_spans[bedrock_spans["span_status_code"] != "OK"]

print(f"Total failed Bedrock invoke spans: {len(failed_bedrock_spans)}")

# print(failed_bedrock_spans.columns)
# show full column width
print("Failed Bedrock Calls: (See span_status_message for reason)")
pd.set_option("display.max_colwidth", None)
display(
    failed_bedrock_spans[
        [
            "span_name",
            "span_status_code",
            "span_status_message",
            # "span_attributes.mlflow.spanInputs",
            # "span_attributes.mlflow.spanOutputs",
            # "span_events", # this can help to understand details
            "tags",
        ]
    ]
)

## Trace analysis to find number of all failed invoke calls

In [ ]:
# Filter to only include spans with name starting with 'BedrockRuntime.invoke_model'
bedrock_spans = traces_exploded[
    traces_exploded["span_name"].str.startswith("BedrockRuntime.invoke_model", na=False)
]
# Group bedrock spans by parent ID to find traces where all bedrock calls failed
bedrock_spans_grouped = (
    bedrock_spans.groupby("span_parent_id")
    .agg(
        {
            "span_status_code": ["count", lambda x: (x != "OK").sum()],
            "span_name": "first",
            "tags": "first",
            "span_status_message": lambda x: list(x),
        }
    )
    .reset_index()
)

# Flatten column names
bedrock_spans_grouped.columns = [
    "span_parent_id",
    "total_bedrock_calls",
    "failed_bedrock_calls",
    "span_name",
    "tags",
    "span_status_messages",
]

# Find parent spans where ALL bedrock calls failed
all_failed_parents = bedrock_spans_grouped[
    bedrock_spans_grouped["total_bedrock_calls"]
    == bedrock_spans_grouped["failed_bedrock_calls"]
]

print(f"Total parent spans with bedrock calls: {len(bedrock_spans_grouped)}")
print(f"Parent spans where ALL bedrock calls failed: {len(all_failed_parents)}")

# Join failed parents with original traces to get more context
failed_with_context = all_failed_parents.merge(
    traces_exploded[
        [
            "span_context.span_id",
            "span_name",
            "span_status_message",
            "span_status_code",
            "urn",
        ]
    ],
    left_on="span_parent_id",
    right_on="span_context.span_id",
    how="left",
    suffixes=("_bedrock", "_parent"),
)

if len(failed_with_context) > 0:
    print("\nParent spans with all failed bedrock calls:")
    pd.set_option("display.max_colwidth", None)
    display(
        failed_with_context[
            [
                "urn",
                "span_parent_id",
                "span_name_parent",
                "span_status_messages",
                "total_bedrock_calls",
                "failed_bedrock_calls",
            ]
        ]
    )
else:
    print("\nNo parent spans found where all bedrock calls failed")

## Trace analysis with failed columm batch descriptions

In [ ]:
import json


column_batch_spans = traces_exploded[
    traces_exploded["span_name"].str.contains("generate_column_batch_descriptions")
].copy()

column_batch_spans["span_outputs"] = column_batch_spans[
    "span_attributes.mlflow.spanOutputs"
].apply(lambda x: json.loads(x) if x is not None and isinstance(x, str) else None)

column_batch_spans["failure_reason"] = column_batch_spans["span_outputs"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)
failed_column_batch_spans = column_batch_spans[
    column_batch_spans["failure_reason"].notna()
][["urn", "failure_reason", "span_context.span_id"]]
failed_column_batch_span_ids = (
    failed_column_batch_spans["span_context.span_id"].drop_duplicates().to_list()
)

failed_bedrock_spans_column_batch = bedrock_spans[
    bedrock_spans["span_parent_id"].isin(failed_column_batch_span_ids)
]

# merge bedrock spans with failed column batch spans
failed_column_batch_bedrock_spans_merged_df = pd.merge(
    failed_bedrock_spans_column_batch,
    failed_column_batch_spans,
    left_on="span_parent_id",
    right_on="span_context.span_id",
    how="left",
    suffixes=("_bedrock", "_column_batch"),
)


dataset_column_batch_failure_reasons = (
    failed_column_batch_bedrock_spans_merged_df.groupby(["urn_column_batch"]).agg(
        failure_reasons=("failure_reason", lambda x: x.value_counts().to_dict())
    )
)

if len(dataset_column_batch_failure_reasons) > 0:
    print("Column Batch Failure Reasons Grouped by Urn")
    pd.set_option("display.max_colwidth", None)
    display(dataset_column_batch_failure_reasons)

    print("\nIndividual Bedrock InvokeModel Span outputs for failed column batch spans")
    display(
        failed_column_batch_bedrock_spans_merged_df.groupby(["failure_reason"]).agg(
            span_outputs=("span_attributes.mlflow.spanOutputs", "first")
        )
    )

## Trace analysis to find total input and output tokens count and AWS SDK retry attempts

In [ ]:
success_traces = bedrock_spans[bedrock_spans["span_status_code"] == "OK"].copy()
success_traces["span_attributes.mlflow.spanOutputs"].head(1)
# Extract response metadata and usage information from successful traces
import json


def extract_response_data(span_output):
    try:
        # Parse the JSON string
        data = json.loads(span_output)

        # Extract retry attempts from ResponseMetadata
        retry_attempts = data.get("ResponseMetadata", {}).get("RetryAttempts", 0)

        stop_reason = data.get("body", {}).get("stop_reason", "")

        # Extract input and output tokens from usage
        input_tokens = data.get("body", {}).get("usage", {}).get("input_tokens", 0)
        cache_creation_input_tokens = (
            data.get("body", {}).get("usage", {}).get("cache_creation_input_tokens", 0)
        )
        cache_read_input_tokens = (
            data.get("body", {}).get("usage", {}).get("cache_read_input_tokens", 0)
        )
        output_tokens = data.get("body", {}).get("usage", {}).get("output_tokens", 0)
        performance_config_latency = data.get("performanceConfigLatency", "standard")

        return (
            retry_attempts,
            input_tokens,
            cache_creation_input_tokens,
            cache_read_input_tokens,
            output_tokens,
            stop_reason,
            performance_config_latency,
        )
    except (json.JSONDecodeError, AttributeError, TypeError):
        return None, None, None


# Apply extraction to successful traces
success_traces[
    [
        "retry_attempts",
        "input_tokens",
        "cache_creation_input_tokens",
        "cache_read_input_tokens",
        "output_tokens",
        "stop_reason",
        "performance_config_latency",
    ]
] = success_traces["span_attributes.mlflow.spanOutputs"].apply(
    lambda x: pd.Series(extract_response_data(x))
)
success_traces["total_input_tokens"] = (
    success_traces["input_tokens"]
    + success_traces["cache_creation_input_tokens"]
    + success_traces["cache_read_input_tokens"]
)
success_traces["has_cache_write"] = success_traces["cache_creation_input_tokens"] > 0
success_traces["has_cache_read"] = success_traces["cache_read_input_tokens"] > 0

# Print value counts for retry attempts
print("\nAWS SDK Retry attempts distribution:")
plot_hist(success_traces, "retry_attempts")


# Print value counts for stop reason
print("\nStop reason distribution:")
print(success_traces["stop_reason"].value_counts().sort_index())

# Print sum of input and output tokens
total_input_tokens = success_traces["total_input_tokens"].sum()
total_output_tokens = success_traces["output_tokens"].sum()

print("\nToken counts:")
print(f"Total input tokens: {total_input_tokens:,}")
print(f"Total output tokens: {total_output_tokens:,}")

plot_hist(success_traces, "total_input_tokens", breakdown_col="has_cache_write")
plot_hist(success_traces, "total_input_tokens", breakdown_col="has_cache_read")
plot_hist(
    success_traces, "cache_creation_input_tokens", breakdown_col="has_cache_write"
)
plot_hist(success_traces, "cache_read_input_tokens", breakdown_col="has_cache_read")
plot_hist(success_traces, "output_tokens")

## Bedrock InvokeModel calls, token-based timing analysis 

In [ ]:
success_traces.columns

In [ ]:
def plot_scatter(df, x_col, y_col, breakdown_col=None, figsize=(10, 6), alpha=0.6):
    """
    Create a scatter plot of two columns with optional coloring by a breakdown column.

    Args:
        df (pd.DataFrame): Input dataframe
        x_col (str): Name of column to plot on x-axis
        y_col (str): Name of column to plot on y-axis
        breakdown_col (str, optional): Name of categorical column to use for coloring
        figsize (tuple, optional): Figure size. Defaults to (10, 6)
        alpha (float, optional): Transparency of points. Defaults to 0.6
    """
    plt.figure(figsize=figsize)

    if breakdown_col is not None:
        # Get unique categories
        categories = df[breakdown_col].unique()

        # Create the scatter plot
        for category in categories:
            subset = df[df[breakdown_col] == category]
            plt.scatter(subset[x_col], subset[y_col], label=category)

    else:
        plt.scatter(df[x_col], df[y_col], alpha=alpha)
    plt.legend()
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(f"{x_col} vs {y_col}")
    plt.grid(True, alpha=0.3)
    plt.show()


# Create scatter plot of input tokens vs generation time
plot_scatter(
    success_traces,
    "total_input_tokens",
    "span_execution_time_ms",
    breakdown_col="performance_config_latency",
)

# Create scatter plot of cache write input tokens vs generation time
plot_scatter(
    success_traces,
    "cache_creation_input_tokens",
    "span_execution_time_ms",
    breakdown_col="performance_config_latency",
)

# Create scatter plot of cache read input tokens vs generation time
plot_scatter(
    success_traces,
    "cache_read_input_tokens",
    "span_execution_time_ms",
    breakdown_col="performance_config_latency",
)

# Create scatter plot of output tokens vs generation time
plot_scatter(
    success_traces,
    "output_tokens",
    "span_execution_time_ms",
    breakdown_col="performance_config_latency",
)

## Compare most expensive invoke calls per dataset with total generation time

In [ ]:
urn_bedrock_span_input_tokens = success_traces.groupby("urn").agg(
    input_tokens_max=("total_input_tokens", "max"),
    output_tokens_max=("output_tokens", "max"),
)

urn_tokens_generation_time_df = pd.merge(
    eval_results_table, urn_bedrock_span_input_tokens, on="urn", how="left"
)

# Create scatter plot of input tokens vs generation time
plot_scatter(urn_tokens_generation_time_df, "input_tokens_max", "generation_time")

# Create scatter plot of output tokens vs generation time
plot_scatter(urn_tokens_generation_time_df, "output_tokens_max", "generation_time")


df_high_generation_time = urn_tokens_generation_time_df[
    urn_tokens_generation_time_df["generation_time"] > 60
]
if len(df_high_generation_time) > 0:
    print(
        f"Total tables with generation time greater than 60 seconds: {len(df_high_generation_time)}"
    )
    display(
        df_high_generation_time[
            [
                "urn",
                "generation_time",
                "total_columns",
                "input_tokens_max",
                "output_tokens_max",
            ]
        ]
    )

## Percentage of described columns for all tables vs small tables vs wide tables

In [ ]:
# Filter to only include tables with has_table_description/score as true
filtered_tables = eval_results_table[
    eval_results_table["has_table_description/score"] == True
]

# Divide filtered dataframe into two sections based on total_columns
large_tables = filtered_tables[filtered_tables["total_columns"] >= 100]
small_tables = filtered_tables[filtered_tables["total_columns"] < 100]

print(f"Total tables with table descriptions: {len(filtered_tables)}")

print(f"Tables with >= 100 columns: {len(large_tables)}")
print(f"Tables with < 100 columns: {len(small_tables)}")

# Calculate statistics for large tables (>= 100 columns)
if len(large_tables) > 0:
    large_min = large_tables["percent_columns_described"].min()
    large_max = large_tables["percent_columns_described"].max()
    large_tables_with_fully_described_columns = large_tables[
        large_tables["percent_columns_described"] == 100
    ]

    print(
        f"\nLarge tables (>= 100 columns) - Percent columns described: {len(large_tables)}"
    )
    print(f"  Min: {large_min:.2f}%")
    print(f"  Max: {large_max:.2f}%")
    print(
        f" Percentage 100% described {100 * len(large_tables_with_fully_described_columns) / len(large_tables):.2f}%"
    )
else:
    print("\nNo large tables found")

# Calculate statistics for small tables (< 100 columns)
if len(small_tables) > 0:
    small_min = small_tables["percent_columns_described"].min()
    small_max = small_tables["percent_columns_described"].max()
    small_tables_with_fully_described_columns = small_tables[
        small_tables["percent_columns_described"] == 100
    ]

    print(
        f"\nSmall tables (< 100 columns) - Percent columns described: {len(small_tables)}"
    )
    print(f"  Min: {small_min:.2f}%")
    print(f"  Max: {small_max:.2f}%")
    print(
        f" Percentage 100% described {100 * len(small_tables_with_fully_described_columns) / len(small_tables):.2f}%"
    )
else:
    print("\nNo small tables found")

# Calculate statistics for all filtered tables
# Calculate statistics for small tables (< 100 columns)
if len(filtered_tables) > 0:
    all_min = filtered_tables["percent_columns_described"].min()
    all_max = filtered_tables["percent_columns_described"].max()
    all_tables_with_fully_described_columns = filtered_tables[
        filtered_tables["percent_columns_described"] == 100
    ]

    print(f"\nAll tables - Percent columns described: {len(filtered_tables)} ")
    print(f"  Min: {all_min:.2f}%")
    print(f"  Max: {all_max:.2f}%")
    print(
        f" Percentage 100% described {100 * len(all_tables_with_fully_described_columns) / len(filtered_tables):.2f}%"
    )
else:
    print("\nNo all tables found")